# 02 — Pattern Calibration

**Purpose:** Detect diffraction order positions across the CMOS detector, fit polynomial curves 
through the peak positions, and save the resulting pattern file.

**When to run:** Only when the detector or optics have been physically reconfigured. 
The existing pattern (`pattern_CMOS_20240305.txt`) is stable for routine use.

**Output:** `pattern_NEWDATE.txt` — 2560 × 29 integer array (cols × orders).

**Based on:** `examples/obsolete/4.2_Pattern_LHD.ipynb`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.signal import savgol_filter
import peakutils
import plotly.graph_objects as go

from echelle_spectra.tools.echelle import Calibrations

%matplotlib inline

## Configuration

In [ ]:
import echelle_spectra

CALIB_DIR = echelle_spectra._config['base_path'] / 'resources/calibration_files'

# Sphere image used to detect order positions (bright, flat-field illumination)
files_cmos = {
    "orders": "pattern_CMOS_20240305.txt",       # existing pattern (only needed for cb init)
    "wavelength": "Th_wavelength_CMOS_20240305.txt",
    "sphr": "sphere_cmos_20240305.sif",
    "bkgr": "sphere_cmos_20240305_bkg.sif",
    "integral": "integrating_sphere.txt",
}

# Where to save the new pattern file
OUTPUT_PATTERN = "pattern_CMOS_NEWDATE.txt"  # change NEWDATE, e.g. 20260515

## Load Integrating Sphere Image

The sphere provides uniform illumination across all orders — ideal for locating peak positions.

In [ ]:
cb = Calibrations(folder=str(CALIB_DIR), filenames=files_cmos)
cb.load_sphere()

# Average all frames
sph = cb.sphr.images.sum(axis=0) / cb.sphr.info['NumberOfFrames']
bkg = cb.bkgr.images.sum(axis=0) / cb.bkgr.info['NumberOfFrames']
img = sph - bkg

print("Sphere image shape:", sph.shape)
print("Background image shape:", bkg.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
norm = mcolors.LogNorm(vmin=max(sph.min(), 1), vmax=sph.max())
ax.imshow(sph, origin="lower", cmap="inferno", norm=norm, aspect="auto")
ax.set_title("Integrating sphere — raw")
ax.set_xlabel("Column (wavelength direction)")
ax.set_ylabel("Row (order direction)")
plt.tight_layout()
plt.show()

## Inspect a Single Vertical Slice

Check that peak detection works at a representative column before running over all columns.

In [ ]:
NROWS, NCOLS = img.shape
xpixels = np.arange(NROWS)

slice_col = 550  # column to inspect
y = img[:, slice_col]
amp = np.exp(3e-3 * xpixels)  # amplify weaker high-order signal
y_smoothed = savgol_filter(y, 21, 1)

plt.figure(figsize=(12, 4))
plt.plot(y, 'k', alpha=0.4, label='raw')
plt.plot(y_smoothed * amp, label='smoothed × amplitude')
plt.xlabel("Row (pixel)")
plt.ylabel("Counts")
plt.title(f"Vertical slice at column {slice_col}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Adjust these parameters until all 29 orders are detected
PEAK_THRESHOLD = 0.13   # relative threshold (0–1)
PEAK_MIN_DIST  = 50     # minimum pixel distance between peaks
POLY_BASELINE  = 6      # polynomial degree for baseline

yprep = savgol_filter(img[:, slice_col], 21, 1) * amp
base = peakutils.baseline(yprep, POLY_BASELINE)
ind = peakutils.indexes(yprep - base, thres=PEAK_THRESHOLD, min_dist=PEAK_MIN_DIST)

fig = go.Figure()
fig.add_trace(go.Scatter(x=xpixels, y=yprep,     mode='lines', name='slice'))
fig.add_trace(go.Scatter(x=ind,      y=yprep[ind], mode='markers', name='peaks', marker=dict(size=8, color='red')))
fig.add_trace(go.Scatter(x=xpixels, y=base,       mode='lines', name='baseline'))
fig.update_layout(template='plotly_white', title=f"Peak detection — column {slice_col} — {len(ind)} peaks found")
fig.show()

print(f"Peaks found: {len(ind)}  (expected: 29 for CMOS)")
print("Peak positions:", ind)

## Detect Peaks Across All Sampled Columns

In [ ]:
def sample_columns(ncols, step_size=150, num_steps=10):
    """Return evenly-spaced column indices centered on the detector."""
    center = ncols // 2
    start  = center - (num_steps // 2 * step_size)
    return np.arange(start, start + num_steps * step_size, step_size)


def detect_peaks_at_column(img, col, xpixels, amp, threshold, min_dist, poly_deg):
    """Return peak row indices for a single column."""
    ysm = savgol_filter(img[:, col], 21, 1) * amp
    base = peakutils.baseline(ysm, poly_deg)
    return peakutils.indexes(ysm - base, thres=threshold, min_dist=min_dist)


ycols = sample_columns(NCOLS, step_size=150, num_steps=10)
print("Sampling columns:", ycols)

peaks = []
for col in ycols:
    ind = detect_peaks_at_column(img, col, xpixels, amp, PEAK_THRESHOLD, PEAK_MIN_DIST, POLY_BASELINE)
    peaks.append(ind)
    print(f"  col {col:4d}: {len(ind)} peaks")

## Manual Correction (if needed)

If any column returned the wrong number of peaks, correct it here.
For example, drop a spurious first peak: `peaks[0] = peaks[0][1:]`

In [ ]:
# Example manual corrections — adjust indices as needed:
# peaks[0] = peaks[0][1:]    # drop first peak at index 0
# peaks[-1] = peaks[-1][:-1] # drop last peak at last index

for i, (col, pk) in enumerate(zip(ycols, peaks)):
    print(f"  col {col:4d}: {len(pk)} peaks — {'OK' if len(pk)==29 else 'CHECK'}")

## Fit Polynomial Curves Through Peak Positions

In [ ]:
def fit_pattern(peaks, ycols, ncols):
    """Fit a degree-2 polynomial through each order's peak positions across sampled columns.
    
    Returns a list of poly1d objects, one per order, evaluated over all columns.
    """
    pat_y = np.array(peaks)
    fits = [np.poly1d(np.polyfit(ycols, pat_y[:, j], 2)) for j in range(pat_y.shape[1])]
    return fits


all_cols = np.arange(NCOLS)
pattern_fits = fit_pattern(peaks, ycols, NCOLS)
print(f"Fitted {len(pattern_fits)} order curves.")

## Verify: Overlay Pattern on Sphere Image

In [ ]:
norm = mcolors.LogNorm(vmin=max(sph.min(), 1), vmax=sph.max())
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(sph, origin="lower", cmap="viridis", norm=norm, aspect="auto")

# Plot detected peaks
for pk_at_col, col in zip(peaks, ycols):
    ax.plot(np.full(len(pk_at_col), col), pk_at_col, '.r', markersize=4)

# Plot fitted curves
for fit in pattern_fits:
    ax.plot(all_cols, fit(all_cols), 'w-', lw=0.8, alpha=0.7)

ax.set_title("Order pattern overlay (white = fits, red = detected peaks)")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

## Save Pattern File

In [ ]:
# Evaluate each fit over all columns to build the integer pattern array
pattern_array = np.column_stack([fit(all_cols).astype(int) for fit in pattern_fits])
print("Pattern array shape:", pattern_array.shape, "  (cols × orders)")

SAVE = False  # set to True to write the file
if SAVE:
    save_path = CALIB_DIR / OUTPUT_PATTERN
    np.savetxt(save_path, pattern_array, fmt='%d')
    print(f"Saved: {save_path}")
else:
    print("SAVE=False — not writing file. Set SAVE=True when satisfied.")